# LSTM – Dataset 1A – LOSO Cross-Validation
Mirrors the data-loading and evaluation pipeline from `run.py`.

In [1]:
import glob
import os
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, Dataset, random_split

In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────────
_HERE        = os.path.abspath('')          # notebook directory
DATASET_PATH = os.path.join(_HERE, '..', '..', '..', 'Processed-DataSets', 'Dataset_1A')
FIGURES_DIR  = os.path.join(_HERE, '..', 'figures')
MODELS_DIR   = _HERE
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Hyper-parameters ───────────────────────────────────────────────────────────
WINDOW_SIZE  = 500
STEP_SIZE    = 250
N_CHANNELS   = 6
N_CLASSES    = 11
BATCH_SIZE   = 32
EPOCHS       = 100
RANDOM_SEED  = 42
VAL_SPLIT    = 0.1
LR           = 1e-3
LR_FACTOR    = 0.5
LR_PATIENCE  = 4
ES_PATIENCE  = 8

torch.manual_seed(RANDOM_SEED)

ACTIVITY_NAMES = {
    1:  'Sitting – Reading',
    2:  'Sitting – Writing',
    3:  'Computer – Typing',
    4:  'Computer – Browsing',
    5:  'Sitting – Moving head/body',
    6:  'Sitting – Moving chair',
    7:  'Stand up from sitting',
    8:  'Standing',
    9:  'Walking',
    10: 'Running',
    11: 'Taking stairs',
}

## 1  Device

In [3]:
if torch.cuda.is_available():
    device = torch.device('cuda:0')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('Device:', device)

Device: mps


## 2  Data loading  (same as `run.py → load_windows`)

In [4]:
def _exp_no(exp_id: int) -> int:
    """Map raw experiment number to activity label 1–11."""
    return exp_id % 11 or 11


def load_windows(window_size: int, step_size: int, dataset_path: str = DATASET_PATH):
    """
    Returns
    -------
    X           : (n_windows, window_size, N_CHANNELS)  float32
    y           : (n_windows,)  int  – activity labels 1–11
    subject_ids : (n_windows,)  int
    """
    all_files = glob.glob(os.path.join(dataset_path, '*', '*.csv'))
    if not all_files:
        raise FileNotFoundError(f'No CSV files found under: {dataset_path}')

    rows = []
    for path in all_files:
        fname  = os.path.basename(path)
        parts  = fname.split('_')
        if len(parts) < 6:
            continue
        try:
            exp_id = int(parts[0])
        except ValueError:
            continue
        sensor      = parts[4]
        user_folder = os.path.basename(os.path.dirname(path))
        try:
            subject_id = int(user_folder.replace('User', ''))
        except ValueError:
            continue
        rows.append({'path': path, 'expID': exp_id, 'sensor': sensor,
                     'subject_id': subject_id})

    meta    = pd.DataFrame(rows)
    windows, labels, subjects = [], [], []
    skipped = 0

    for exp_id, group in meta.groupby('expID'):
        if not {'Accelerometer', 'Gyroscope'}.issubset(set(group['sensor'].values)):
            skipped += 1
            continue

        acc_path   = group.loc[group['sensor'] == 'Accelerometer', 'path'].iloc[0]
        gyr_path   = group.loc[group['sensor'] == 'Gyroscope',     'path'].iloc[0]
        subject_id = group['subject_id'].iloc[0]

        acc = pd.read_csv(acc_path, usecols=['x-axis (g)',     'y-axis (g)',     'z-axis (g)'])
        gyr = pd.read_csv(gyr_path, usecols=['x-axis (deg/s)', 'y-axis (deg/s)', 'z-axis (deg/s)'])

        n_samples = min(len(acc), len(gyr))
        raw = np.concatenate([
            acc.values[:n_samples].astype(np.float32),
            gyr.values[:n_samples].astype(np.float32),
        ], axis=1)  # (n_samples, 6)

        label = _exp_no(exp_id)
        for start in range(0, n_samples - window_size + 1, step_size):
            windows.append(raw[start: start + window_size])
            labels.append(label)
            subjects.append(subject_id)

    X           = np.array(windows,  dtype=np.float32)
    y           = np.array(labels,   dtype=int)
    subject_ids = np.array(subjects, dtype=int)

    print(f'Loaded {X.shape[0]} windows  |  window={window_size}  step={step_size}')
    print(f'Experiments: {meta["expID"].nunique()}  |  Subjects: {len(np.unique(subject_ids))}')
    print(f'Skipped (missing sensor): {skipped}')
    return X, y, subject_ids


X, y, subject_ids = load_windows(WINDOW_SIZE, STEP_SIZE)
print('X:', X.shape, '  y:', y.shape, '  subjects:', np.unique(subject_ids))

Loaded 5744 windows  |  window=500  step=250
Experiments: 129  |  Subjects: 12
Skipped (missing sensor): 0
X: (5744, 500, 6)   y: (5744,)   subjects: [ 1  2  3  4  5  6  7  8  9 10 11 12]


## 3  Dataset wrapper

In [5]:
class WindowDataset(Dataset):
    """Wraps (n, W, C) windows as sequences – no flattening for LSTM."""

    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## 4  LSTM model

In [6]:
class LSTMClassifier(nn.Module):
    """
    3-layer stacked LSTM classifier.
    Input shape: (batch, window_size, n_channels)
    """

    def __init__(self, n_channels: int = 6, n_classes: int = 11,
                 hidden: int = 128, dropout: float = 0.5):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_channels,
            hidden_size=hidden,
            num_layers=3,
            batch_first=True,
            dropout=dropout,
        )
        self.drop = nn.Dropout(dropout)
        self.fc1  = nn.Linear(hidden, 64)
        self.relu = nn.ReLU()
        self.fc2  = nn.Linear(64, n_classes)

    def forward(self, x):
        # x: (batch, time, channels)
        _, (h_n, _) = self.lstm(x)   # h_n: (num_layers, batch, hidden)
        out = self.drop(h_n[-1])     # last layer: (batch, hidden)
        out = self.relu(self.fc1(out))
        return self.fc2(out)


def build_model(n_channels: int, n_classes: int) -> nn.Module:
    return LSTMClassifier(n_channels=n_channels, n_classes=n_classes)


# Quick architecture check
_m = build_model(N_CHANNELS, N_CLASSES)
print(_m)
total_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

LSTMClassifier(
  (lstm): LSTM(6, 128, num_layers=3, batch_first=True, dropout=0.5)
  (drop): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=11, bias=True)
)
Trainable parameters: 342,795


## 5  Training utilities

In [7]:
def _train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
        correct    += (logits.argmax(1) == y_batch).sum().item()
        n          += len(y_batch)
    return total_loss / n, correct / n


@torch.no_grad()
def _evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logits = model(X_batch)
        total_loss += criterion(logits, y_batch).item() * len(y_batch)
        correct    += (logits.argmax(1) == y_batch).sum().item()
        n          += len(y_batch)
    return total_loss / n, correct / n


def fit(model, X_train: np.ndarray, y_train: np.ndarray,
        device: torch.device, model_path: str):
    """Train with validation split, early stopping, and LR scheduling."""
    dataset = WindowDataset(X_train, y_train)
    val_len = max(1, int(len(dataset) * VAL_SPLIT))
    trn_len = len(dataset) - val_len
    trn_set, val_set = random_split(
        dataset, [trn_len, val_len],
        generator=torch.Generator().manual_seed(RANDOM_SEED),
    )
    trn_loader = DataLoader(trn_set, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, factor=LR_FACTOR, patience=LR_PATIENCE
    )

    best_val_loss = float('inf')
    no_improve    = 0
    history = {'trn_loss': [], 'trn_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(1, EPOCHS + 1):
        trn_loss, trn_acc = _train_one_epoch(model, trn_loader, criterion, optimizer, device)
        val_loss, val_acc = _evaluate(model, val_loader, criterion, device)
        scheduler.step(val_loss)
        history['trn_loss'].append(trn_loss)
        history['trn_acc'].append(trn_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if epoch % 10 == 0 or epoch == 1:
            print(f'  epoch {epoch:3d}  '
                  f'loss {trn_loss:.4f}  acc {trn_acc:.4f}  '
                  f'val_loss {val_loss:.4f}  val_acc {val_acc:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_path + '.tmp')
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= ES_PATIENCE:
                print(f'  Early stopping at epoch {epoch}')
                break

    model.load_state_dict(torch.load(model_path + '.tmp', map_location=device))
    return history

## 6  LOSO cross-validation

In [ ]:
le    = LabelEncoder()
y_enc = le.fit_transform(y)

unique_subjects  = np.unique(subject_ids)
fold_accuracies  = []
all_y_true       = []
all_y_pred       = []
best_acc         = -1.0
model_path       = os.path.join(MODELS_DIR, f'lstm_loso_best_W{WINDOW_SIZE}_S{STEP_SIZE}.pth')

_, W, C = X.shape

for subject in unique_subjects:
    test_mask  = subject_ids == subject
    train_mask = ~test_mask

    X_train_raw = X[train_mask]
    X_test_raw  = X[test_mask]
    y_train     = y_enc[train_mask]
    y_test      = y_enc[test_mask]

    # Per-channel StandardScaler fitted on train, applied to test
    scaler = StandardScaler()
    n_tr   = X_train_raw.shape[0]
    X_train_sc = scaler.fit_transform(
        X_train_raw.reshape(-1, C)
    ).reshape(n_tr, W, C).astype(np.float32)
    X_test_sc = scaler.transform(
        X_test_raw.reshape(-1, C)
    ).reshape(X_test_raw.shape[0], W, C).astype(np.float32)

    print('─' * 60)
    print(f'LOSO fold – held-out: User{subject}  '
          f'(test={test_mask.sum()}  train={train_mask.sum()})')

    model = build_model(C, N_CLASSES).to(device)
    fit(model, X_train_sc, y_train, device, model_path)

    model.eval()
    with torch.no_grad():
        inp    = torch.from_numpy(X_test_sc).float().to(device)
        y_pred = model(inp).argmax(1).cpu().numpy()

    acc = accuracy_score(y_test, y_pred)
    fold_accuracies.append(acc)
    all_y_true.extend(y_test.tolist())
    all_y_pred.extend(y_pred.tolist())
    print(f'  User{subject} accuracy: {acc:.4f}')

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), model_path)
        print(f'  ✓ New best model saved (acc={acc:.4f})')

# Clean up temp checkpoint
_tmp = model_path + '.tmp'
if os.path.exists(_tmp):
    os.remove(_tmp)

all_y_true = np.array(all_y_true)
all_y_pred = np.array(all_y_pred)

────────────────────────────────────────────────────────────
LOSO fold – held-out: User1  (test=328  train=5416)
  epoch   1  loss 2.2193  acc 0.1506  val_loss 2.0413  val_acc 0.1756
  epoch  10  loss 1.8538  acc 0.2636  val_loss 1.9050  val_acc 0.2699
  epoch  20  loss 1.6029  acc 0.3169  val_loss 1.5812  val_acc 0.3290
  epoch  30  loss 1.3690  acc 0.4238  val_loss 1.2153  val_acc 0.4806
  epoch  40  loss 1.4393  acc 0.4248  val_loss 1.1814  val_acc 0.4750
  epoch  50  loss 1.2355  acc 0.4823  val_loss 1.1151  val_acc 0.5379
  epoch  60  loss 1.0855  acc 0.5225  val_loss 0.9727  val_acc 0.5970
  epoch  70  loss 0.9987  acc 0.5598  val_loss 0.9436  val_acc 0.5786
  epoch  80  loss 0.9594  acc 0.5760  val_loss 0.8987  val_acc 0.6155
  epoch  90  loss 0.8872  acc 0.5971  val_loss 0.8743  val_acc 0.6266


## 7  Results

In [ ]:
print('=' * 60)
print('LOSO SUMMARY')
print('=' * 60)
for subject, acc in zip(unique_subjects, fold_accuracies):
    print(f'  User{subject}: {acc:.4f}')
print(f'  Mean accuracy : {np.mean(fold_accuracies):.4f}')
print(f'  Std  accuracy : {np.std(fold_accuracies):.4f}')
print()

class_names = [ACTIVITY_NAMES[le.classes_[i]] for i in range(len(le.classes_))]
print('=' * 60)
print('CLASSIFICATION REPORT (all folds combined)')
print('=' * 60)
print(classification_report(all_y_true, all_y_pred, target_names=class_names))

In [ ]:
# Confusion matrix
tag = f'W{WINDOW_SIZE}_S{STEP_SIZE}'
cm  = confusion_matrix(all_y_true, all_y_pred)

plt.figure(figsize=(13, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'LSTM – Confusion Matrix (LOSO)  [{tag}]', fontsize=14)
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
cm_path = os.path.join(FIGURES_DIR, f'lstm_confusion_matrix_{tag}.png')
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'Saved: {cm_path}')

In [ ]:
# Per-subject accuracy bar chart
plt.figure(figsize=(10, 5))
plt.bar([f'User{s}' for s in unique_subjects], fold_accuracies,
        color='steelblue', edgecolor='white')
plt.axhline(np.mean(fold_accuracies), color='tomato', linewidth=1.5,
            linestyle='--', label=f'Mean {np.mean(fold_accuracies):.3f}')
plt.ylabel('Accuracy')
plt.title(f'LSTM – Per-subject LOSO Accuracy  [{tag}]')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
bar_path = os.path.join(FIGURES_DIR, f'lstm_per_subject_accuracy_{tag}.png')
plt.savefig(bar_path, dpi=150)
plt.show()
print(f'Saved: {bar_path}')